In [1]:
from pathlib import Path
PAIR_JSON = Path("data/train_set_fixed.json")
CSV_PATH = Path("data/all_real_with_split_fixed.csv")
OUT = Path("data/train_targets.json")

print("pair json:", PAIR_JSON.exists(), "csv:", CSV_PATH.exists())

pair json: True csv: True


In [2]:
from pathlib import Path
PAIR_JSON = Path("data/train_set_fixed.json")
CSV_PATH = Path("data/all_real_with_split_fixed.csv")
OUT = Path("data/train_targets.json")

print("pair json:", PAIR_JSON.exists(), "csv:", CSV_PATH.exists())

pair json: True csv: True


In [3]:
import json, re
import pandas as pd
from pathlib import Path

def extract_id(path):
    if not isinstance(path, str):
        raise ValueError("path not a string")
    m = re.search(r"/(id\d+)/", path)
    if m:
        return m.group(1)
    parts = Path(path).parts
    for p in parts[::-1]:
        if re.match(r"id\d+", p):
            return p
    # fallback try numeric chunk
    m2 = re.search(r"/(0*\d{3,6})/", path)
    if m2:
        return "id" + m2.group(1).lstrip("0")
    raise ValueError(f"Could not extract id from path: {path}")

def make_targets(pair_json_path, csv_path, out_json):
    df = pd.read_csv(csv_path)
    id_to_split = df.set_index("source_id")["split"].to_dict()

    with open(pair_json_path, "r") as f:
        data = json.load(f)
        items = data["data"] if isinstance(data, dict) and "data" in data else data

    targets = []
    for item in items:
        tgt = item.get("file_path") or item.get("target_file")
        if not tgt:
            continue
        try:
            sid = extract_id(tgt)
        except Exception:
            sid = item.get("id") or "unknown"
        split = id_to_split.get(sid, item.get("split", "train"))
        fake_label = int(item.get("fake_label", item.get("fake", 1)))
        targets.append({
            "file_path": tgt,
            "id": sid,
            "split": split,
            "fake_label": fake_label
        })

    with open(out_json, "w") as f:
        json.dump(targets, f, indent=2)
    print(f"Wrote {len(targets)} entries to {out_json}")


In [4]:
if not PAIR_JSON.exists():
    raise FileNotFoundError(PAIR_JSON)

make_targets(str(PAIR_JSON), str(CSV_PATH), str(OUT))
print("Done. Output at:", OUT)

Wrote 14003 entries to data/train_targets.json
Done. Output at: data/train_targets.json
